# 07 - Tournament editions and weather windows
Produces tennis_clean,
tournament_editions: one row per tournament edition, with coordinates and the weather window to fetch
Run after 06_venues

## Install the pinned libraries

In [ ]:

# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v1"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "07_tournament_editions"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location=REGION)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
DATED = f"{PROJECT_ID}.tennis_clean.matches_dated"
NAMEMAP = f"{PROJECT_ID}.tennis_clean.venue_name_map"
LOCATIONS = f"{PROJECT_ID}.tennis_clean.venue_locations"
EDITIONS = f"{PROJECT_ID}.tennis_clean.tournament_editions"

## Build the editions table
One row per tournament edition (one year of one tournament), with its coordinates. This table is the list of weather requests.

The weather window comes from the estimated match days (notebook 05).

In [ ]:
editions = run(f"""
SELECT tourney_id,
       ANY_VALUE(name_key) AS name_key, COUNT(DISTINCT name_key) AS n_names,
       ANY_VALUE(tourney_level) AS tourney_level, COUNT(DISTINCT tourney_level) AS n_levels,
       ANY_VALUE(draw_size) AS draw_size,
       ANY_VALUE(event_format) AS event_format,
       MIN(tourney_date) AS start_date, COUNT(DISTINCT tourney_date) AS n_dates,
       MAX(match_date_max) AS last_match_day,
       COUNT(*) AS n_matches
FROM `{DATED}` GROUP BY tourney_id
""")
# Each edition must have exactly one name, level and start date
assert (editions[["n_names", "n_levels", "n_dates"]] == 1).all().all()

name_map = run(f"SELECT name_key, search_name FROM `{NAMEMAP}`")
locations = run(f"SELECT search_name, place, country_code, latitude, longitude, elevation_m, timezone FROM `{LOCATIONS}`")
editions = editions.merge(name_map, on="name_key", how="left").merge(locations, on="search_name", how="left")
assert editions["latitude"].notna().all(), "Some editions have no location"

for c in ["start_date", "last_match_day"]:
    editions[c] = pd.to_datetime(editions[c])
editions["end_date"] = editions["last_match_day"] + pd.Timedelta(days=1)      # one day of padding

# The weather archive lags by about 5 days, so never ask for later dates
latest_ok = pd.Timestamp.now().normalize() - pd.Timedelta(days=6)
print("windows cut back for the archive delay:", int((editions["end_date"] > latest_ok).sum()))
editions["end_date"] = editions["end_date"].clip(upper=latest_ok)

editions = editions[["tourney_id", "name_key", "search_name", "place", "country_code", "tourney_level",
                     "draw_size", "event_format", "start_date", "end_date", "last_match_day",
                     "latitude", "longitude", "elevation_m", "timezone", "n_matches"]
                    ].sort_values("tourney_id").reset_index(drop=True)

## Checks


In [ ]:
n_expected = run(f"SELECT COUNT(DISTINCT tourney_id) AS n FROM `{DATED}`")["n"][0]
print(len(editions), "editions | expected:", n_expected, "| tourney_id unique:", editions["tourney_id"].is_unique)
assert len(editions) == n_expected and editions["tourney_id"].is_unique
assert (editions["end_date"] >= editions["last_match_day"]).all(), "A match day falls outside its weather window"

editions["window_days"] = (editions["end_date"] - editions["start_date"]).dt.days + 1
print(editions.groupby("event_format")["window_days"].agg(["count", "min", "max"]))
print(editions[["start_date", "end_date"]].agg(["min", "max"]))
print("total days to fetch:", editions["window_days"].sum())
editions.sample(5, random_state=SEED)[["tourney_id", "place", "country_code", "event_format", "start_date", "end_date"]]

## Save the table (BigQuery) and record its version
BigQuery is the table's one home: it is no longer also copied as a CSV file to the bucket. A manifest records its row count, content fingerprint, inputs, and the archive cut-off date used (it depends on the day the notebook is run).

In [ ]:
from google.cloud import storage
import json

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

to_save = editions.drop(columns=["window_days"]).copy()
for c in ["start_date", "end_date", "last_match_day"]:
    to_save[c] = to_save[c].dt.date

bq.load_table_from_dataframe(
    to_save, EDITIONS,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
print("BigQuery rows:", bq.get_table(EDITIONS).num_rows)

out = table_version(bq, EDITIONS)
parents = [table_version(bq, t) for t in [DATED, NAMEMAP, LOCATIONS]]
manifest_path = f"manifests/tournament_editions_{utc_stamp()}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"table": out, "parents": parents, "archive_cutoff": str(latest_ok.date()), **RUN_INFO}, indent=1),
    content_type="application/json")
print("Manifest saved:", manifest_path)
register(bucket, "tennis_clean.tournament_editions", out["fingerprint"], outputs=[out],
         parents=parents, manifest_path=manifest_path)